# Giai đoạn 1: Thu thập và Làm sạch Dữ liệu
**Dự án:** Phân tích và Dự đoán Tỷ lệ Tội phạm

Nội dung notebook này bao gồm:
1. Tải dữ liệu từ UCI Repository bằng thư viện `ucimlrepo`.
2. Gộp các tập dữ liệu thành phần (IDs, Features, Targets).
3. Kiểm tra cấu trúc dữ liệu và xử lý các giá trị thiếu.
4. Xuất file dữ liệu sạch (`cleaned_crime_data.csv`) chuẩn bị cho bước phân t.

In [70]:
# !pip install ucimlrepo pandas numpy
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

### 1. Tải dữ liệu từ UCI

In [61]:
# Tải bộ dữ liệu Communities and Crime Unnormalized (ID = 211)
crime_dataset = fetch_ucirepo(id=211)

# Trích xuất các DataFrame thành phần
X = crime_dataset.data.features
y = crime_dataset.data.targets
ids = crime_dataset.data.ids

print(f'Cấu trúc ban đầu: Features {X.shape}, Targets {y.shape}, IDs {ids.shape}')

Cấu trúc ban đầu: Features (2215, 125), Targets (2215, 18), IDs (2215, 4)


### 2. Gộp tất cả các cột thành một DataFrame duy nhất

In [62]:
# Gộp theo trục cột (axis=1) để giữ lại toàn bộ 147 biến
df_raw = pd.concat([ids, X, y], axis=1)
print(f'Tổng kích thước DataFrame tổng hợp: {df_raw.shape}')

df_raw= df_raw.rename(columns={"communityname": "communityname"})

Tổng kích thước DataFrame tổng hợp: (2215, 147)


0       11980
1       23123
2       29344
3       16656
4       11245
        ...  
2210    56216
2211    12251
2212    32824
2213    13547
2214    28898
Name: pop, Length: 2215, dtype: int64

### 3. Xoá dòng bị trùng lặp

In [63]:
df_raw = df_raw.drop_duplicates()
df_raw.shape

(2215, 147)

### 4. Chuyển đổi kiểu dữ liệu

In [64]:
cols_to_convert = df_raw.columns[5:]

for col in cols_to_convert:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

In [65]:
print("Phân tích dữ liệu ban đầu:")
df_raw.describe()

Phân tích dữ liệu ban đầu:


,countyCode,communityCode,fold,pop,perHoush,pctBlack,pctWhite,pctAsian,pctHisp,pct12-21,...,burglaries,burglPerPop,larcenies,larcPerPop,autoTheft,autoTheftPerPop,arsons,arsonsPerPop,violentPerPop,nonViolPerPop
count,994.000000,991.000000,2215.000000,2.215000e+03,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,...,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2124.000000,2124.000000,1994.000000,2118.000000
mean,65.587525,45209.251261,5.494357,5.311798e+04,2.707327,9.335102,83.979819,2.670203,7.950176,14.445837,...,761.236890,1033.430203,2137.629295,3372.979150,516.692586,473.965628,30.907721,32.153682,589.078922,4908.241804
std,117.831399,25425.861573,2.872924,2.046203e+05,0.334120,14.247156,16.419080,4.473843,14.589832,4.518623,...,3111.702756,763.354442,7600.573464,1901.316145,3258.164244,504.666026,180.125248,39.240900,614.784518,2739.708901
min,1.000000,70.000000,1.000000,1.000500e+04,1.600000,0.000000,2.680000,0.030000,0.120000,4.580000,...,2.000000,16.920000,10.000000,77.860000,1.000000,6.550000,0.000000,0.000000,0.000000,116.790000
25%,11.000000,22887.000000,3.000000,1.436600e+04,2.500000,0.860000,76.320000,0.620000,0.930000,12.250000,...,95.000000,511.690000,392.000000,2040.080000,30.000000,156.952500,1.000000,7.670000,161.700000,2918.070000
50%,27.000000,46925.000000,5.000000,2.279200e+04,2.660000,2.870000,90.350000,1.230000,2.180000,13.620000,...,205.000000,822.715000,747.000000,3079.510000,75.000000,302.355000,5.000000,21.080000,374.060000,4425.450000
75%,80.500000,65805.000000,8.000000,4.302400e+04,2.850000,11.145000,96.225000,2.670000,7.810000,15.360000,...,508.000000,1350.232500,1675.000000,4335.410000,232.500000,589.775000,16.000000,42.852500,794.400000,6229.280000
max,840.000000,94597.000000,10.000000,7.322564e+06,5.280000,96.670000,99.630000,57.460000,95.290000,54.400000,...,99207.000000,11881.020000,235132.000000,25910.550000,112464.000000,4968.590000,5119.000000,436.370000,4877.060000,27119.760000


### 5. Xử lý giá trị thiếu


In [66]:
# Tính toán tỷ lệ phần trăm giá trị thiếu trên từng cột
missing_matrix = df_raw.isnull().mean() * 100
missing_cols = missing_matrix[missing_matrix > 0].sort_values(ascending=False)

print(f'Số lượng cột bị thiếu dữ liệu: {len(missing_cols)}')
print('Top 10 cột có tỷ lệ thiếu cao nhất (%):')
print(missing_cols.head(10))

Số lượng cột bị thiếu dữ liệu: 41
Top 10 cột có tỷ lệ thiếu cao nhất (%):
policCarsAvail      84.514673
gangUnit            84.514673
policOperBudget     84.514673
policAveOT          84.514673
numDiffDrugsSeiz    84.514673
officDrugUnits      84.514673
pctPolicMinority    84.514673
pctPolicAsian       84.514673
pctPolicHisp        84.514673
pctPolicBlack       84.514673
dtype: float64


In [67]:
# Loại bỏ các cột có tỷ lệ thiếu > 50% vì chúng không đủ thông tin để phân tích
threshold = 50.0
cols_to_drop = missing_matrix[missing_matrix > threshold].index
df_filtered = df_raw.drop(columns=cols_to_drop)

print(f'Đã loại bỏ {len(cols_to_drop)} cột có tỷ lệ thiếu > {threshold}%')
print(f'Các cột bị loại bỏ: {list(cols_to_drop)}')
print(f'Kích thước mới sau khi lọc cột: {df_filtered.shape}')


Đã loại bỏ 24 cột có tỷ lệ thiếu > 50.0%
Các cột bị loại bỏ: ['countyCode', 'communityCode', 'numPolice', 'policePerPop', 'policeField', 'policeFieldPerPop', 'policeCalls', 'policCallPerPop', 'policCallPerOffic', 'policePerPop2', 'racialMatch', 'pctPolicWhite', 'pctPolicBlack', 'pctPolicHisp', 'pctPolicAsian', 'pctPolicMinority', 'officDrugUnits', 'numDiffDrugsSeiz', 'policAveOT', 'policCarsAvail', 'policOperBudget', 'pctPolicPatrol', 'gangUnit', 'policBudgetPerPop']
Kích thước mới sau khi lọc cột: (2215, 123)


In [68]:
# Đối với các cột còn lại thiếu dữ liệu <50%, điền bằng giá trị trung vị (median)
for col in df_filtered.columns:
    if df_filtered[col].dtype in [np.float64, np.int64]:
        median_val = df_filtered[col].median()
        df_filtered[col] = df_filtered[col].fillna(median_val)

print(f'Số lượng ô trống còn lại sau khi điền Median: {df_filtered.isnull().sum().sum()}')

Số lượng ô trống còn lại sau khi điền Median: 0


### 6. Lưu dữ liệu sạch

In [71]:
df_filtered.to_csv('cleaned_crime_data.csv', index=False)
print('Đã lưu thành công file "cleaned_crime_data.csv"!')

Đã lưu thành công file "cleaned_crime_data.csv"!
